#data.yaml
train: C:\Users\<DATASET-URL-LOCAL>\train\images

val: C:\Users\<DATASET-URL-LOCAL>\valid\images

test: C:\Users\<DATASET-URL-LOCAL>\test\images

nc: 3
names:
- person
- weapon
- umbrella

In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

#drive.mount('/content/drive')






Mounted at /content/drive


In [ ]:
!pip install ultralytics
#INSTALL THE LIBRARY



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 10.8 MB/s eta 0:00:00


In [ ]:
import os
import zipfile
import shutil # Added for copying directories

drive_path = '/content/drive/MyDrive'
local_extract_dir = '/content/dataset'

found_dataset_source = None
dataset_type = None # To track if it's a 'folder' or 'zip'

# Define potential parent directories to search within for the dataset
# Based on the error output, 'YOLOv8_Weapon_Detection' is the only item in MyDrive
potential_parent_dirs = [
    drive_path,
    os.path.join(drive_path, 'YOLOv8_Weapon_Detection')
]

# --- Step 1: Search for an existing 'dataset_combined' folder ---
for parent_dir in potential_parent_dirs:
    dataset_folder_path = os.path.join(parent_dir, 'dataset_combined')
    if os.path.isdir(dataset_folder_path):
        found_dataset_source = dataset_folder_path
        dataset_type = 'folder'
        print(f"Found dataset as a directory: '{found_dataset_source}'")
        break

# --- Step 2: If no folder found, search for a 'dataset_combined.zip' file ---
if found_dataset_source is None:
    for parent_dir in potential_parent_dirs:
        # List files in the parent_dir and search for the zip
        for f in os.listdir(parent_dir):
            if 'dataset_combined' in f.lower() and f.endswith('.zip'):
                zip_file_name = f
                found_dataset_source = os.path.join(parent_dir, zip_file_name)
                dataset_type = 'zip'
                print(f"Found dataset as a ZIP file: '{found_dataset_source}'")
                break
        if found_dataset_source is not None:
            break

# --- Step 3: Handle the found dataset or raise an error ---
if found_dataset_source is None:
    # Provide more specific paths checked for clarity
    searched_paths_str = ', '.join(potential_parent_dirs)
    raise FileNotFoundError(
        f"Could not find a directory named 'dataset_combined' or a zip file containing 'dataset_combined' "
        f"in any of the searched locations: {searched_paths_str}. "
        f"Available items directly in {drive_path}: {os.listdir(drive_path)[:10]}"
    )

os.makedirs(local_extract_dir, exist_ok=True)

if dataset_type == 'folder':
    print(f"Copying dataset from '{found_dataset_source}' to '{local_extract_dir}'...")
    # Use shutil.copytree to copy the entire directory structure
    # dirs_exist_ok=True allows copying into an existing destination directory
    shutil.copytree(found_dataset_source, os.path.join(local_extract_dir, os.path.basename(found_dataset_source)), dirs_exist_ok=True)
    print(f"✅ Dataset successfully copied to local disk: '{os.path.join(local_extract_dir, os.path.basename(found_dataset_source))}'!")

elif dataset_type == 'zip':
    print(f"Extracting ZIP file from '{found_dataset_source}' to '{local_extract_dir}'...")
    with zipfile.ZipFile(found_dataset_source, 'r') as zip_ref:
        zip_ref.extractall(local_extract_dir)
    print(f"✅ Dataset successfully unzipped to local disk: '{local_extract_dir}'!")


Found dataset as a ZIP file: '/content/drive/MyDrive/dataset_combined.zip'
Extracting ZIP file from '/content/drive/MyDrive/dataset_combined.zip' to '/content/dataset'...
✅ Dataset successfully unzipped to local disk: '/content/dataset'!


In [ ]:
import os
import yaml

# The local directory where the dataset is expected to be extracted/copied.
# This matches the `local_extract_dir` from the previous cell.
local_extract_base_dir = '/content/dataset'

# The dataset (e.g., 'dataset_combined') is expected to be a subdirectory
# within `local_extract_base_dir` after extraction/copying.
dataset_root = os.path.join(local_extract_base_dir, 'dataset_combined')

# Verify if the expected dataset root exists.
# If the previous step (_xrDFLHCE8Q1) failed, this directory will not exist.
if not os.path.exists(dataset_root):
    raise FileNotFoundError(
        f"The dataset directory '{dataset_root}' does not exist. "
        f"Please ensure the dataset (either a 'dataset_combined' folder or a zip file) "
        f"has been successfully placed in your Google Drive and extracted/copied to '{local_extract_base_dir}'."
    )

data_yaml = {
    'path': dataset_root,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 3,
    'names': ['person', 'weapon', 'umbrella']
}

# Save the updated config for Colab
yaml_path = '/content/data_colab.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f)

print(f"✅ Dataset YAML successfully created at: {yaml_path}, pointing to: {dataset_root}")


✅ Dataset YAML successfully created at: /content/data_colab.yaml, pointing to: /content/dataset/dataset_combined


In [ ]:
from ultralytics import YOLO
import torch

# Load pre-trained YOLOv8 model (yolov8m.pt or yolov8m.pt)
model = YOLO("yolov8m.pt")

# Check for CUDA availability
if not torch.cuda.is_available():
    print("WARNING: CUDA is not available. Training will proceed on CPU. Please ensure your Colab runtime type is set to GPU (e.g., T4 GPU).")

# Start training
results = model.train(
    data="/content/data_colab.yaml",
    epochs=50,             # Adjust epochs based on dataset size
    imgsz=640,             # Standard image resolution (640-balanced)(1280-more accurate but slower)
    batch=16,              # Standard batch size for T4 GPU
    device=0,              # Use GPU (device=0) if available
    project="/content/drive/MyDrive/YOLOv8_Weapon_Detection", # Save outputs directly to Drive
    name="yolov8_run"
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.162 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, free

In [ ]:
from ultralytics import YOLO

# 1. Load your trained model weights from Google Drive
model_path = '/content/drive/MyDrive/YOLOv8_Weapon_Detection/yolov8_run-3/weights/best.pt'  # Update subfolder path if stored in a specific folder inside MyDrive
model = YOLO(model_path)

# 2. Validate model on the test split using the yaml file
metrics = model.val(data='/content/data_colab.yaml', split='test')

# Print summary metrics
print(f"mAP50-95: {metrics.box.map}")
print(f"mAP50:    {metrics.box.map50}")
print(f"Precision: {metrics.box.p.mean()}")
print(f"Recall:    {metrics.box.r.mean()}")

'''# 3. Run prediction on the test images folder
results = model.predict(
    source="/content/dataset/dataset_combined/test/images",
    save=True,
    conf=0.25
)'''

#3. Run prediction on the test image folder and saving the output permanently
results = model.predict(
    source=os.path.join(dataset_root, 'test/images'),
    save=True,
    conf=0.25,
    project='/content/drive/MyDrive/YOLO_Outputs',  # Main Drive directory
    name='test_results'                             # Subfolder name
)


Ultralytics 8.4.164 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 92 layers, 25,841,497 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 778.4±500.7 MB/s, size: 41.3 KB)
val: Scanning /content/dataset/dataset_combined/test/labels.cache... 276 images, 21 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 276/276 68.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.0it/s 9.0s
                   all        276        474       0.88      0.627      0.787      0.493
                person        230        390      0.963      0.675      0.826      0.561
                weapon         74         78      0.843      0.538      0.689      0.398
              umbrella          6          6      0.833      0.667      0.845       0.52
Speed: 3.2ms preprocess, 20.8ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/runs/detec

In [ ]:
'''The accuracy metrics for your model are as follows:

mAP50-95: 0.4933
mAP50: 0.7868
Precision: 0.8798
Recall: 0.6266'''

### Object Detection Performance Metrics

- **mAP (mean Average Precision)**: This is the most common metric for object detection. It is the mean of the Average Precisions (AP) across all classes. AP is calculated by finding the area under the precision-recall curve for each class.
  - **mAP50**: mAP calculated at an Intersection Over Union (IoU) threshold of 0.5. This means a detected bounding box is considered a True Positive if its IoU with a ground truth bounding box is 0.5 or higher.
  - **mAP50-95**: mAP averaged over different IoU thresholds, from 0.5 to 0.95 with a step size of 0.05. This provides a more comprehensive evaluation of detection performance across varying localization accuracies.

- **Precision**: The proportion of true positive detections among all positive detections (true positives + false positives). High precision means fewer false alarms.

- **Recall**: The proportion of true positive detections among all actual positive instances (true positives + false negatives). High recall means fewer missed detections.